# Mamba3 LLM

### Stuff

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from einops import rearrange
import os
import json

In [ ]:
def apply_rotary(x: torch.Tensor, angles: torch.Tensor) -> torch.Tensor:
    *batch, N, R = x.shape
    x = x.reshape(*batch, N // 2, 2, R)
    x1, x2 = x[..., 0, :], x[..., 1, :]
    cos = angles.cos().unsqueeze(-1)
    sin = angles.sin().unsqueeze(-1)
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-2)
    return out.reshape(*batch, N, R)


class Mamba3(nn.Module):
    def __init__(
        self,
        d_model,
        n_head,
        d_state,
        rank,
        expand=2,
        A_floor=1e-4,
        min_dt=0.001,
        max_dt=0.1,
        dt_init_floor=1e-4,
        use_pre_gate_norm=True,
        chunk_size=32,
    ):
        super().__init__()
        self.d_state = d_state
        self.d_model = d_model
        self.n_head = n_head
        self.rank = rank
        self.A_floor = A_floor
        self.min_dt = min_dt
        self.max_dt = max_dt
        self.dt_init_floor = dt_init_floor
        self.use_pre_gate_norm = use_pre_gate_norm
        self.chunk_size = chunk_size

        assert d_state % 2 == 0, "d_state must be even for complex SSM"
        assert (d_model * expand) % n_head == 0, "(d_model * expand) must be divisible by n_head"

        self.head_dim = (d_model * expand) // n_head

        # Projections
        self.Z = nn.Linear(d_model, n_head * self.head_dim, bias=False)
        self.X = nn.Linear(d_model, n_head * self.head_dim, bias=False)
        self.B = nn.Linear(d_model, d_state * rank, bias=False)
        self.C = nn.Linear(d_model, d_state * rank, bias=False)
        self.dt = nn.Linear(d_model, n_head, bias=False)
        self.A = nn.Linear(d_model, n_head, bias=False)
        self.trap = nn.Sequential(nn.Linear(d_model, n_head, bias=False), nn.Sigmoid())
        self.angle_rate = nn.Linear(d_model, n_head * (d_state // 2), bias=False)

        # MIMO & SSM parameters
        self.z_scale = nn.Parameter(torch.ones(n_head, self.head_dim, rank))
        self.x_scale = nn.Parameter(torch.ones(n_head, self.head_dim, rank))
        self.o_scale = nn.Parameter(torch.ones(n_head, self.head_dim, rank) / rank)

        self.Bnorm = nn.RMSNorm([d_state], eps=1e-6)
        self.Cnorm = nn.RMSNorm([d_state], eps=1e-6)
        self.B_bias = nn.Parameter(torch.ones(n_head, d_state))
        self.C_bias = nn.Parameter(torch.ones(n_head, d_state))
        self.B_bias._no_weight_decay = True
        self.C_bias._no_weight_decay = True

        self.Dskip = nn.Parameter(torch.ones(1, 1, n_head, self.head_dim, rank))
        self.Dskip._no_weight_decay = True
        self.pre_gate_norm = nn.RMSNorm([self.head_dim, self.rank], eps=1e-6)

        _dt = torch.exp(
            torch.rand(self.n_head, dtype=torch.float32) * (math.log(max_dt) - math.log(min_dt))
            + math.log(min_dt)
        ).clamp(min=dt_init_floor)
        _dt_bias = _dt + torch.log(-torch.expm1(-_dt))
        self.dt_bias = nn.Parameter(_dt_bias)
        self.dt_bias._no_weight_decay = True

        # Output projection
        self.out = nn.Linear(self.n_head * self.head_dim, d_model, bias=True)

        # Causal mask for intra-chunk SSD
        self.register_buffer(
            "chunk_tril",
            torch.tril(torch.ones(chunk_size, chunk_size, dtype=torch.bool)),
            persistent=False,
        )

    def chunked_ssm(self, X, B_rot, C_rot, log_alpha, beta, gamma, h0, Bprev0):
        Bsz, T, H, P, R = X.shape
        Cs = self.chunk_size

        Y = X.new_empty(Bsz, T, H, P, R)
        h = h0
        Bprev = Bprev0

        for start in range(0, T, Cs):
            end = min(start + Cs, T)
            clen = end - start

            X_c = X[:, start:end]
            B_c = B_rot[:, start:end]
            C_c = C_rot[:, start:end]
            log_a_c = log_alpha[:, start:end]
            b_c = beta[:, start:end]
            g_c = gamma[:, start:end]

            V = torch.einsum("bchnr,bchpr->bchnp", B_c, X_c)
            V_prev = torch.cat([Bprev.to(V.dtype).unsqueeze(1), V[:, :-1]], dim=1)
            V_p = V.permute(0, 2, 1, 3, 4)
            Vp_p = V_prev.permute(0, 2, 1, 3, 4)
            b_e = b_c.permute(0, 2, 1).unsqueeze(-1).unsqueeze(-1)
            g_e = g_c.permute(0, 2, 1).unsqueeze(-1).unsqueeze(-1)
            U = b_e * Vp_p + g_e * V_p

            log_a = log_a_c.float().permute(0, 2, 1)
            cum_log = log_a.cumsum(-1)
            diff = cum_log.unsqueeze(-1) - cum_log.unsqueeze(-2)
            tril = self.chunk_tril[:clen, :clen]
            decay = torch.exp(diff.masked_fill(~tril, -float("inf"))).to(U.dtype)

            H_intra = torch.einsum("bhts,bhsnp->bhtnp", decay, U)

            h_coef = torch.exp(cum_log).unsqueeze(-1).unsqueeze(-1).to(H_intra.dtype)
            H = H_intra + h_coef * h.unsqueeze(2).to(H_intra.dtype)

            C_p = C_c.permute(0, 2, 1, 3, 4)
            Y_c = torch.einsum("bhtnp,bhtnr->bhtpr", H, C_p)
            Y[:, start:end] = Y_c.permute(0, 2, 1, 3, 4).to(Y.dtype)

            h = H[:, :, -1].to(h.dtype)
            Bprev = V[:, -1].to(Bprev.dtype)

        return Y, h, Bprev

    def forward(self, input, state=None):
        BATCH_SIZE, SEQLEN, _ = input.shape
        init_h, init_Bprev, init_angle = state if state is not None else (None, None, None)

        # Project input
        Z = self.Z(input)
        X = self.X(input)
        B = self.B(input)
        C = self.C(input)
        dt = F.softplus(self.dt(input) + self.dt_bias)
        A = -F.softplus(self.A(input)).clamp(min=self.A_floor)
        trap = self.trap(input)
        angle_rate = self.angle_rate(input)

        # Reshape and expand MIMO rank R
        Z = Z.view(BATCH_SIZE, SEQLEN, self.n_head, self.head_dim)
        X = X.view(BATCH_SIZE, SEQLEN, self.n_head, self.head_dim)
        Z = F.silu(Z.unsqueeze(-1) * self.z_scale)
        X = X.unsqueeze(-1) * self.x_scale

        # BCNorm + head-specific channel-wise biases
        B = B.view(BATCH_SIZE, SEQLEN, self.d_state, self.rank)
        C = C.view(BATCH_SIZE, SEQLEN, self.d_state, self.rank)
        B = self.Bnorm(B.transpose(-2, -1)).transpose(-2, -1)
        C = self.Cnorm(C.transpose(-2, -1)).transpose(-2, -1)

        B_bias = self.B_bias.view(1, 1, self.n_head, self.d_state, 1)
        C_bias = self.C_bias.view(1, 1, self.n_head, self.d_state, 1)
        B = B.unsqueeze(2) + B_bias
        C = C.unsqueeze(2) + C_bias

        # Data-dependent RoPE angles
        angle_rate = angle_rate.view(BATCH_SIZE, SEQLEN, self.n_head, self.d_state // 2)
        angle_increment = angle_rate * dt.unsqueeze(-1)
        angles = angle_increment.cumsum(dim=1)
        if init_angle is not None:
            angles = angles + init_angle.unsqueeze(1)
        angles = torch.fmod(angles, 2.0 * math.pi)
        angle_out = angles[:, -1]

        B_rot = apply_rotary(B, angles)
        C_rot = apply_rotary(C, angles)

        # Exponential-trapezoidal coefficients
        log_alpha = dt * A
        alpha = torch.exp(log_alpha)
        beta = (1.0 - trap) * dt * alpha
        gamma = trap * dt

        h = (
            init_h
            if init_h is not None
            else torch.zeros(
                BATCH_SIZE,
                self.n_head,
                self.d_state,
                self.head_dim,
                device=input.device,
                dtype=input.dtype,
            )
        )
        Bprev = init_Bprev if init_Bprev is not None else torch.zeros_like(h)

        y, h, Bprev = self.chunked_ssm(X, B_rot, C_rot, log_alpha, beta, gamma, h, Bprev)

        y = y + (self.Dskip * X)
        if self.use_pre_gate_norm:
            y = self.pre_gate_norm(y)
        y = y * Z

        y = (y * self.o_scale).sum(dim=-1)
        y = rearrange(y, "b t h p -> b t (h p)")
        return self.out(y), (h, Bprev, angle_out)

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, d_model, n_head, ff_dim, dropout=0.1, max_cache_len=2048):
        super().__init__()
        assert d_model % n_head == 0, "d_model must be divisible by n_head"
        self.d_model = d_model
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.dropout = dropout
        self.max_cache_len = max_cache_len

        self.norm1 = nn.RMSNorm([d_model], eps=1e-6)
        self.norm2 = nn.RMSNorm([d_model], eps=1e-6)

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.attn_out = nn.Linear(d_model, d_model, bias=False)

        self.do1 = nn.Dropout(dropout)
        self.do2 = nn.Dropout(dropout)
        self.ff = nn.Linear(d_model, ff_dim * 2)
        self.out = nn.Linear(ff_dim, d_model)

    def forward(self, input, state=None):
        B, T, D = input.shape
        x = self.norm1(input)

        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, H, T, Dh)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if state is not None and state[0] is not None:
            k_cache, v_cache = state
            k = torch.cat([k_cache, k], dim=2)
            v = torch.cat([v_cache, v], dim=2)
            if k.size(2) > self.max_cache_len:
                k = k[:, :, -self.max_cache_len :]
                v = v[:, :, -self.max_cache_len :]

        S = k.size(2)
        past_len = S - T
        if past_len == 0:
            attn_out = F.scaled_dot_product_attention(
                q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0
            )
        else:
            row_idx = torch.arange(T, device=input.device).unsqueeze(1) + past_len
            col_idx = torch.arange(S, device=input.device).unsqueeze(0)
            causal_mask = col_idx <= row_idx
            attn_out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=causal_mask, dropout_p=self.dropout if self.training else 0.0
            )

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D)
        x = input + self.do1(self.attn_out(attn_out))

        residual = x
        x = self.norm2(x)
        gate, val = self.ff(x).chunk(2, dim=-1)
        x = self.out(F.silu(gate) * val)
        x = residual + self.do2(x)
        return x, (k, v)


class Mamba3Block(nn.Module):
    def __init__(
        self,
        ff_dim,
        d_model,
        n_head,
        d_state,
        rank,
        chunk_size=32,
        expand=2,
        A_floor=1e-4,
        min_dt=0.001,
        max_dt=0.1,
        dt_init_floor=1e-4,
        dropout=0.1,
    ):
        super().__init__()
        self.norm1 = nn.RMSNorm([d_model], eps=1e-6)
        self.norm2 = nn.RMSNorm([d_model], eps=1e-6)
        self.mamba = Mamba3(
            d_model,
            n_head,
            d_state,
            rank,
            expand=expand,
            A_floor=A_floor,
            min_dt=min_dt,
            max_dt=max_dt,
            dt_init_floor=dt_init_floor,
            chunk_size=chunk_size,
        )
        self.do1 = nn.Dropout(dropout)
        self.do2 = nn.Dropout(dropout)
        self.ff = nn.Linear(d_model, ff_dim * 2)
        self.out = nn.Linear(ff_dim, d_model)

    def forward(self, input, state=None):
        x = self.norm1(input)
        m, new_state = self.mamba(x, state=state)
        x = input + self.do1(m)

        residual = x
        x = self.norm2(x)
        gate, val = self.ff(x).chunk(2, dim=-1)
        x = self.out(F.silu(gate) * val)
        x = residual + self.do2(x)
        return x, new_state


class Mamba3LLM(nn.Module):
    def __init__(
        self,
        n_layer,
        vocab_size,
        ff_dim,
        d_model,
        n_head,
        d_state,
        rank,
        chunk_size=32,
        attn_every=5,
        expand=2,
        A_floor=1e-4,
        min_dt=0.001,
        max_dt=0.1,
        dt_init_floor=1e-4,
        dropout=0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)
        blocks = []
        for i in range(n_layer):
            if attn_every > 0 and (i + 1) % attn_every == 0:
                blocks.append(AttentionBlock(d_model, n_head, ff_dim, dropout=dropout))
            else:
                blocks.append(
                    Mamba3Block(
                        ff_dim,
                        d_model,
                        n_head,
                        d_state,
                        rank,
                        expand=expand,
                        A_floor=A_floor,
                        min_dt=min_dt,
                        max_dt=max_dt,
                        dt_init_floor=dt_init_floor,
                        dropout=dropout,
                        chunk_size=chunk_size,
                    )
                )
        self.blocks = nn.ModuleList(blocks)
        self.post_norm = nn.RMSNorm([d_model], eps=1e-6)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, input, return_state=False, layer_states=None):
        x = self.embedding(input)
        new_states = []
        for idx, block in enumerate(self.blocks):
            prev_state = None if layer_states is None else layer_states[idx]
            x, st = block(x, state=prev_state)
            new_states.append(st)

        x = self.post_norm(x)
        logits = self.out(x)
        if return_state:
            return logits, new_states
        return logits

In [ ]:
def detach_layer_states(layer_states):
    if layer_states is None:
        return None
    return [
        tuple(t.detach() if t is not None else None for t in st) if st is not None else None
        for st in layer_states
    ]

In [ ]:
def count_parameters(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

In [ ]:
def tokenize_chats(tokenizer, chats, add_generation_prompt=False):
    encoded = tokenizer.apply_chat_template(
        chats, tokenize=True, add_generation_prompt=add_generation_prompt
    )
    if isinstance(encoded, dict) or hasattr(encoded, "input_ids"):
        return encoded["input_ids"]
    return encoded

In [ ]:
def build_Optimizer(model, lr=3e-4, weight_decay=0.01):
    decay_params = []
    no_decay_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if (
            getattr(p, "_no_weight_decay", False)
            or p.ndim <= 1
            or "bias" in name
            or "norm" in name
            or "scale" in name
        ):
            no_decay_params.append(p)
        else:
            decay_params.append(p)

    return AdamW(
        [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=lr,
    )

### Train stream

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_targets = [
    ("HuggingFaceFW/fineweb-edu","default","train")
]
TEST_SIZE = 100
MAX_DATA_SIZE = 500000 + TEST_SIZE
BATCH_SIZE = 8
CHUNK_SIZE = 64
MAX_NUM_CHUNKS = 8
GRAD_ACCUMULATION_STEPS = 1
SAVE_EVERY_STEP = 500
LOG_EVERY_STEP = 100
TOTAL_STEP = int((MAX_DATA_SIZE-TEST_SIZE) * len(dataset_targets) / BATCH_SIZE)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("nvidia/Nemotron-Cascade-2-30B-A3B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = len(tokenizer)

In [ ]:
model = Mamba3LLM(
    n_layer=32,
    attn_every=4,
    chunk_size=16,
    vocab_size=VOCAB_SIZE,
    ff_dim=6144,
    d_model=2048,
    n_head=16,
    d_state=128,
    rank=4,
    expand=2,
    dropout=0.0
).to(device)
trainable_full, total_full = count_parameters(model)
print(f"Parameters — trainable: {trainable_full:,}  |  total: {total_full:,}")
print(f"Approx size (fp32): {total_full * 4 / 1e9:.2f} GB")

In [ ]:
optimizer = build_Optimizer(model, lr=3e-4, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer,T_max=TOTAL_STEP)
scaler = torch.amp.GradScaler(device)

In [ ]:
checkpoint_path = "./mamba3llm_100.pt"
start_step = 0

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_step = checkpoint['step']
    print(f"Successfully resumed training from step {start_step}!")
    trainable_full, total_full = count_parameters(model)
    print(f"Parameters — trainable: {trainable_full:,}  |  total: {total_full:,}")
    print(f"Approx size (fp32): {total_full * 4 / 1e9:.2f} GB")
else:
    print("No checkpoint found. Starting training from scratch.")
torch.set_float32_matmul_precision("high")
model = torch.compile(model, dynamic=True, mode="default")

In [ ]:
model.train()
optimizer.zero_grad()

datasets = []
for name,config,split in dataset_targets:
    print(f"Collecting {name} - {config}")
    ds = load_dataset(name,config,split=split,streaming=True)
    ds = iter(ds)
    datasets.append(ds)

In [ ]:
def get_iter(d):
    try:
        return next(d)
    except:
        return None

@torch.no_grad()
def inference(model, prompt, max_token_generation=50):
    model.eval()
    tokens = tokenizer(prompt)['input_ids']
    inp = torch.tensor([tokens], dtype=torch.long, device=device)

    with torch.amp.autocast(device.type):
        logits, layer_states = model(inp, return_state=True, layer_states=None)
        next_token = logits[0, -1, :].argmax(dim=-1).item()
        tokens.append(next_token)

    for _ in range(max_token_generation - 1):
        if next_token == tokenizer.eos_token_id:
            break
        inp = torch.tensor([[next_token]], dtype=torch.long, device=device)
        with torch.amp.autocast(device.type):
            logits, layer_states = model(inp, return_state=True, layer_states=layer_states)
            next_token = logits[0, -1, :].argmax(dim=-1).item()
            tokens.append(next_token)

    return tokenizer.decode(tokens)

@torch.no_grad()
def evaluate_model(model, data, limit_num_chunk=1):
    model.eval()
    total_loss = 0.0
    total_chunks = 0
    for input_id in data:
        if len(input_id) < 2:
            continue
        full_in = input_id[:-1]
        full_tgt = input_id[1:]
        layer_states = None
        counter = 0
        for i in range(0, len(full_in), CHUNK_SIZE):
            if counter >= limit_num_chunk:
                break
            inp = torch.tensor([full_in[i : i + CHUNK_SIZE]], dtype=torch.long, device=device)
            tgt = torch.tensor([full_tgt[i : i + CHUNK_SIZE]], dtype=torch.long, device=device)
            with torch.amp.autocast(device.type):
                logits, layer_states = model(
                    inp, return_state=True, layer_states=layer_states
                )
                loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), tgt.view(-1))
            total_loss += loss.item()
            total_chunks += 1
            counter += 1

    avg_loss = total_loss / max(1, total_chunks)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    return avg_loss, perplexity

def log_result(step, train_res, val_res, acc_loss):
    data = {}
    if os.path.isfile("log.json"):
        try:
            with open("log.json", "r") as f:
                data = json.loads(f.read())
        except Exception:
            data = {}

    data[str(step)] = {
        "train_loss": train_res,
        "val_loss": val_res,
        "accumulated_loss": acc_loss,
    }
    with open("log.json", "w") as f:
        f.write(json.dumps(data, indent=2))

def get_log_result(step):
    if os.path.isfile("log.json"):
        try:
            with open("log.json", "r") as f:
                data = json.loads(f.read())
            return data.get(str(step), None)
        except Exception:
            return None
    return None

In [ ]:
test_chats = []
for _ in range(TEST_SIZE):
    for ds in datasets:
        dt = get_iter(ds)
        if dt is not None:
            test_chats.append(dt["text"])
test_input_ids = tokenizer(test_chats)['input_ids']
print("Test size:", len(test_input_ids))

In [ ]:
step = 0
ds_index = 0
accumulated_loss = 0
if os.path.exists('log.json'):
    dd = get_log_result("latest")
    if dd is not None:
        accumulated_loss = dd['accumulated_loss']

print(f"TOTAL_STEP: {TOTAL_STEP} - BATCH_SIZE: {BATCH_SIZE} - TOTAL_DATA: {MAX_DATA_SIZE * len(dataset_targets)} - ACC_LOSS: {accumulated_loss}")

while step < TOTAL_STEP:
    batch_data = []
    for _ in range(BATCH_SIZE):
        dt = None
        for _ in range(len(datasets)):
            dt = get_iter(datasets[ds_index])
            if dt is None:
                ds_index = (ds_index + 1) % len(datasets)
            else:
                break
        if dt is None:
            raise ValueError(f"[{step + 1}/{TOTAL_STEP}] No more data to train")
        batch_data.append(dt["text"])
        ds_index = (ds_index + 1) % len(datasets)

    step += 1
    if step <= start_step:
        continue
    print(f"[{step}/{TOTAL_STEP}] Current step")

    input_ids_list = tokenizer(batch_data)['input_ids']
    max_len = min(max(len(x) for x in input_ids_list), CHUNK_SIZE * MAX_NUM_CHUNKS + 1)

    padded_inputs = []
    padded_labels = []
    for seq in input_ids_list:
        seq = seq[:max_len]
        pad_len = max_len - len(seq)
        padded_inputs.append(seq + [tokenizer.pad_token_id] * pad_len)
        padded_labels.append(seq + [-100] * pad_len)

    input_tensor = torch.tensor(padded_inputs, dtype=torch.long, device=device)
    label_tensor = torch.tensor(padded_labels, dtype=torch.long, device=device)

    full_inputs = input_tensor[:, :-1]
    full_targets = label_tensor[:, 1:]
    total_seq_len = full_inputs.size(1)

    valid_chunk_indices = [
        i
        for i in range(0, total_seq_len, CHUNK_SIZE)[:MAX_NUM_CHUNKS]
        if (full_targets[:, i : i + CHUNK_SIZE] != -100).any()
    ]
    num_chunks = max(1, len(valid_chunk_indices))
    layer_states = None

    for i in valid_chunk_indices:
        chunk_in = full_inputs[:, i : i + CHUNK_SIZE]
        chunk_tgt = full_targets[:, i : i + CHUNK_SIZE]

        with torch.amp.autocast(device.type):
            logits, layer_states = model(
                chunk_in, return_state=True, layer_states=layer_states
            )
            layer_states = detach_layer_states(layer_states)

            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                chunk_tgt.reshape(-1),
                ignore_index=-100,
            )
            scaled_loss = loss / (num_chunks * GRAD_ACCUMULATION_STEPS)

        scaler.scale(scaled_loss).backward()
        accumulated_loss += loss.item() / num_chunks

    if step % GRAD_ACCUMULATION_STEPS == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

    if step % (SAVE_EVERY_STEP * GRAD_ACCUMULATION_STEPS) == 0:
        raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
        ckpt_payload = {
            "step": step,
            "model_state_dict": raw_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }
        step_ckpt_path = f"./mamba3llm_{step}.pt"
        torch.save(ckpt_payload, step_ckpt_path)
        log_result("latest", 0, 0, accumulated_loss)
        print(f"Checkpoint saved to {step_ckpt_path}")

    if step % (LOG_EVERY_STEP * GRAD_ACCUMULATION_STEPS) == 0:
        current_lr = scheduler.get_last_lr()[0]
        tl = accumulated_loss / (LOG_EVERY_STEP * GRAD_ACCUMULATION_STEPS)
        print(f"[{step}/{TOTAL_STEP}] Train Loss: {tl:.4f} | LR: {current_lr:.6f}")

        val_loss, val_perplexity = evaluate_model(model, test_input_ids)
        print(
            f"[{step}/{TOTAL_STEP}] Val Loss: {val_loss:.4f} | "
            f"Val Perplexity: {val_perplexity:.4f}"
        )
        test_out = inference(model, "The capital of France is")
        print("-" * 50)
        print(test_out)
        print("-" * 50)
        model.train()

        log_result(step, tl, val_loss, accumulated_loss)
        accumulated_loss = 0.0